# 01 — Model validation

**Workflow version:** 0.5.0

Validate the released input artifacts before any phenotype benchmark is interpreted. This notebook checks file identity, package/solver provenance, model dimensions, objective, wild-type growth, and the paper's 1% viability threshold.

In [ ]:
from pathlib import Path
import hashlib
import json
import platform
import sys

import pandas as pd
import cobra
from cobra.io import read_sbml_model

SOLVER = "glpk"

cwd = Path.cwd().resolve()
ROOT = cwd.parent if cwd.name == "notebooks" else cwd
DATA_DIR = ROOT / "data" / "raw"
RESULTS_DIR = ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

ORIGINAL_XML = DATA_DIR / "yeast9.0.xml"
CURATED_XML = DATA_DIR / "Yeast9_curated.xml"
DATASET_XLSX = DATA_DIR / "mmc3.xlsx"

required = [ORIGINAL_XML, CURATED_XML, DATASET_XLSX]
missing = [path for path in required if not path.exists()]
if missing:
    raise FileNotFoundError("Missing local inputs: " + ", ".join(str(path) for path in missing))

if SOLVER not in cobra.util.solver.solvers:
    raise RuntimeError(f"Solver {SOLVER!r} is unavailable. Available: {sorted(cobra.util.solver.solvers)}")

print("Repository root:", ROOT)
print("Python:", sys.version.split()[0])
print("COBRApy:", cobra.__version__)
print("Solver:", SOLVER)

In [ ]:
def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while chunk := handle.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()

input_manifest = pd.DataFrame([
    {
        "file": path.name,
        "bytes": path.stat().st_size,
        "sha256": sha256_file(path),
    }
    for path in required
])

display(input_manifest)

In [ ]:
original = read_sbml_model(str(ORIGINAL_XML))
curated = read_sbml_model(str(CURATED_XML))
original.solver = SOLVER
curated.solver = SOLVER

def model_summary(label, model):
    objective = [
        (reaction.id, float(reaction.objective_coefficient))
        for reaction in model.reactions
        if reaction.objective_coefficient != 0
    ]
    growth = model.slim_optimize(error_value=float("nan"))
    return {
        "model": label,
        "reactions": len(model.reactions),
        "metabolites": len(model.metabolites),
        "genes": len(model.genes),
        "objective": objective,
        "solver_status": str(model.solver.status),
        "wt_growth": float(growth),
    }

qc = pd.DataFrame([
    model_summary("Yeast9", original),
    model_summary("Yeast9_curated", curated),
])

display(qc)

In [ ]:
expected_objective = [("r_2111", 1.0)]
if qc.loc[qc["model"] == "Yeast9", "objective"].iloc[0] != expected_objective:
    raise AssertionError("Unexpected Yeast9 biomass objective.")

if not qc["solver_status"].str.lower().eq("optimal").all():
    raise RuntimeError("At least one wild-type optimization was not optimal.")

wt_original = float(qc.loc[qc["model"] == "Yeast9", "wt_growth"].iloc[0])
viability_threshold = 0.01 * wt_original

print(f"Yeast9 WT growth: {wt_original:.12g}")
print(f"1% viability threshold: {viability_threshold:.12g}")

In [ ]:
qc.to_csv(RESULTS_DIR / "01_model_qc.csv", index=False)
input_manifest.to_csv(RESULTS_DIR / "01_input_manifest.csv", index=False)

manifest = {
    "workflow_version": "0.5.0",
    "python": sys.version,
    "python_executable": sys.executable,
    "platform": platform.platform(),
    "cobra": cobra.__version__,
    "solver": SOLVER,
    "viability_fraction": 0.01,
    "viability_threshold": viability_threshold,
    "files": input_manifest.to_dict(orient="records"),
}

(RESULTS_DIR / "01_run_manifest.json").write_text(
    json.dumps(manifest, indent=2), encoding="utf-8"
)

print("Saved model validation outputs to:", RESULTS_DIR)

## Interpretation checkpoint

The paper reports a default Yeast9 growth of approximately **0.0859** and defines inviability as growth below **1%** of that value. Do not continue to the phenotype benchmark if model identity, objective, solver status, or wild-type growth are inconsistent with the released model.